# Robot-arm training: Colab platform preflight

Select **Runtime → Change runtime type → GPU**, then run these cells interactively.

**This account uses the free tier.** Expect a T4-class 16 GB GPU, ~12 h maximum
sessions, idle disconnects and no guaranteed allocation. Free managed runtimes
also disallow SSH shells, remote desktops, web-UI driving and distributed
workers, so an actor/learner split is not an option here.

This notebook installs nothing, mounts no Drive, fetches no weights and uploads
no data. It reports the allocated hardware and runs 20 tiny synthetic optimizer
steps plus a checkpoint round-trip. It is a **compute smoke test**, not task
training: it does not qualify MolmoAct2, LeRobot, MuJoCo/EGL, private assets or
real hardware. Full fine-tuning of MolmoAct2 (5.44B parameters, ~10.1 GiB of
fp16 weights plus ~40 GiB of AdamW state) cannot fit a 16 GB T4 at any batch
size, so the pilot needs an adapter or frozen-backbone path.


In [ ]:
import json, platform, shutil, subprocess, sys, time
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. Select a GPU runtime; do not silently use CPU.")
report = {
    "schema": 1,
    "kind": "synthetic-compute-smoke-only",
    "python": sys.version,
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "platform": platform.platform(),
    "gpu": torch.cuda.get_device_name(0),
    "capability": list(torch.cuda.get_device_capability(0)),
}
free, total = torch.cuda.mem_get_info()
report.update(vram_total_gib=total / 2**30, vram_free_gib=free / 2**30,
              disk_free_gib=shutil.disk_usage("/content" if Path("/content").exists() else ".").free / 2**30)
report["nvidia_smi"] = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total,memory.free", "--format=csv,noheader"],
    text=True).strip()
report["kernel_log_status"] = "not inspected; managed runtime visibility unknown"
print(json.dumps(report, indent=2))


In [ ]:
# Pure arithmetic: what fits. No torch, no model download.
PARAMS = 5_442_196_272          # allenai/MolmoAct2-SO100_101, HF safetensors total
GIB = 1024 ** 3

def gib(n):
    return round(n / GIB, 2)

weights = {
    "fp32": PARAMS * 4,
    "fp16/bf16": PARAMS * 2,
    "int8": PARAMS * 1,
    "4-bit": PARAMS / 2,
}
adamw = PARAMS * 8              # two fp32 moments
for name, size in weights.items():
    print(f"{name:10s} weights {gib(size):6.2f} GiB")
print(f"{'adamw':10s} state   {gib(adamw):6.2f} GiB (full fine-tune only)")

vram = torch.cuda.get_device_properties(0).total_memory
name = torch.cuda.get_device_name(0)
print(f"\nGPU: {name}  {gib(vram):.2f} GiB")
print("bf16 supported:", torch.cuda.is_bf16_supported())
print("full fine-tune fits:", (weights['fp16/bf16'] + adamw) < 0.6 * vram)
print("adapter/LoRA plausible:", weights['fp16/bf16'] < 0.65 * vram)


In [ ]:
# A bounded, deterministic small-model optimizer/checkpoint smoke test.
# No simulator data, images, pretrained weights or credentials are used.
torch.manual_seed(20261030)
device = "cuda:0"
model = torch.nn.Sequential(torch.nn.Linear(6, 64), torch.nn.Tanh(), torch.nn.Linear(64, 6)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
x = torch.randn(32, 6, device=device)
y = 0.5 * x
losses = []
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
started = time.monotonic()
for step in range(20):
    optimizer.zero_grad(set_to_none=True)
    loss = torch.nn.functional.mse_loss(model(x), y)
    if not torch.isfinite(loss):
        raise RuntimeError(f"nonfinite loss at step {step}")
    loss.backward()
    if not all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters()):
        raise RuntimeError(f"nonfinite gradient at step {step}")
    optimizer.step()
    losses.append(loss.item())
torch.cuda.synchronize()
report.update(optimizer_steps=20, elapsed_seconds=time.monotonic() - started,
              peak_allocated_gib=torch.cuda.max_memory_allocated() / 2**30,
              initial_loss=losses[0], final_loss=losses[-1])
checkpoint_path = Path("/content" if Path("/content").exists() else ".") / "robot-arm-preflight-checkpoint.pt"
expected = model(x).detach()
torch.save({"model":model.state_dict(), "optimizer":optimizer.state_dict(), "step":20}, checkpoint_path)
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
restored = torch.nn.Sequential(torch.nn.Linear(6, 64), torch.nn.Tanh(), torch.nn.Linear(64, 6)).to(device)
restored.load_state_dict(checkpoint["model"])
restored_optimizer = torch.optim.AdamW(restored.parameters(), lr=1e-3)
restored_optimizer.load_state_dict(checkpoint["optimizer"])
if checkpoint["step"] != 20 or not torch.equal(expected, restored(x).detach()):
    raise RuntimeError("checkpoint reload changed predictions or step count")
# Also take one resumed optimizer step to exercise restored optimizer state.
restored_optimizer.zero_grad(set_to_none=True)
resume_loss = torch.nn.functional.mse_loss(restored(x), y)
resume_loss.backward()
if not torch.isfinite(resume_loss) or not all(p.grad is None or torch.isfinite(p.grad).all() for p in restored.parameters()):
    raise RuntimeError("nonfinite resumed optimizer step")
restored_optimizer.step()
if not all(torch.isfinite(p).all() for p in restored.parameters()):
    raise RuntimeError("nonfinite resumed parameters")
report.update(checkpoint_reload_exact=True, resumed_optimizer_step=True,
              checkpoint_bytes=checkpoint_path.stat().st_size,
              passed=True, scope="CUDA/synthetic optimizer only; task training NOT qualified")
report_path = checkpoint_path.with_name("robot-arm-colab-preflight.json")
report_path.write_text(json.dumps(report, indent=2) + "\n")
print(json.dumps(report, indent=2))
print(f"Saved {report_path}; share this report to continue platform qualification.")


## Next gate (not automatically executed)

Return the JSON report to the robot-arm project session. On a free-tier T4 the
next step is **adapter training**, not a full fine-tune: LoRA/QLoRA over a frozen
backbone, micro-batch 1 with gradient accumulation, gradient checkpointing and
checkpoints written to persistent storage so each session is disposable.

Export only the audited training split. The public `XvKuoMing/so101_chess`
dataset is already published; uploading private recordings, camera frames, CAD
or checkpoints to Drive still needs Brian's explicit decision.

A 20-step actual-task pilot with save/reload comes before any long run. No cloud
transfer of private data is implied by this smoke test.
